In [1]:
%pip install numpy pandas matplotlib seaborn scikit-learn

  Using cached numpy-2.4.4-cp313-cp313-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached pandas-3.0.2-cp313-cp313-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached matplotlib-3.10.8-cp313-cp313-macosx_11_0_arm64.whl.metadata (52 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached scikit_learn-1.8.0-cp313-cp313-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached contourpy-1.3.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pillow-12.2.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (8.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached scipy-1.17.1-cp313-cp313-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached numpy-2.4.4-cp313-cp313-macosx_14_0_arm64.whl (5.2 MB)
Using cached pandas-3.0.2-cp313-cp3

In [8]:
datasets = ['./csv/allergies.csv',
            './csv/careplans.csv',
            './csv/claims_transactions.csv',
            './csv/claims.csv',
            './csv/devices.csv',
            './csv/encounters.csv',
            './csv/imaging_studies.csv',
            './csv/immunizations.csv',
            './csv/medications.csv',
            './csv/observations.csv',
            './csv/organizations.csv',
            './csv/patients.csv',
            './csv/procedures.csv',
            './csv/payer_transitions.csv',
            './csv/providers.csv',
            './csv/supplies.csv',]

In [20]:
import pandas as pd
import numpy as np

# ── Load core tables ────────────────────────────────────────────────────────
allergies = pd.read_csv('./csv/allergies.csv', on_bad_lines='skip', engine='python')
careplans = pd.read_csv('./csv/careplans.csv', on_bad_lines='skip', engine='python')
claims_transactions = pd.read_csv('./csv/claims_transactions.csv', on_bad_lines='skip', engine='python')
claims = pd.read_csv('./csv/claims.csv', on_bad_lines='skip', engine='python')
conditions = pd.read_csv('./csv/conditions.csv', on_bad_lines='skip', engine='python')
devices = pd.read_csv('./csv/devices.csv', on_bad_lines='skip', engine='python')
encounters = pd.read_csv('./csv/encounters.csv', on_bad_lines='skip', engine='python')
imaging_studies = pd.read_csv('./csv/imaging_studies.csv', on_bad_lines='skip', engine='python')
immunizations = pd.read_csv('./csv/immunizations.csv', on_bad_lines='skip', engine='python')
medications = pd.read_csv('./csv/medications.csv', on_bad_lines='skip', engine='python')
observations = pd.read_csv('./csv/observations.csv', on_bad_lines='skip', engine='python')
organizations = pd.read_csv('./csv/organizations.csv', on_bad_lines='skip', engine='python')
patients = pd.read_csv('./csv/patients.csv', on_bad_lines='skip',   engine='python')
payer_transitions = pd.read_csv('./csv/payer_transitions.csv', on_bad_lines='skip', engine='python')
procedures = pd.read_csv('./csv/procedures.csv', on_bad_lines='skip', engine='python')
providers = pd.read_csv('./csv/providers.csv', on_bad_lines='skip', engine='python')
supplies = pd.read_csv('./csv/supplies.csv', on_bad_lines='skip', engine='python')
print("All tables loaded successfully.")

All tables loaded successfully.


# questions to ponder about:

why will current start from 2021 and not before? or after?

In [ ]:
# ── Temporal split: historical vs current ─────────────────────────────────────
CUT_OFF_DATE = pd.Timestamp('2021-01-01', tz='UTC')

# Event tables are split on their relevant date columns.
# Static lookup tables do not have event timestamps, so they are shared across both halves.
DATE_COLUMNS = {
    'allergies': 'START',
    'careplans': 'START',
    'claims_transactions': 'FROMDATE',
    'claims': 'SERVICEDATE',
    'conditions': 'START',
    'devices': 'START',
    'encounters': 'START',
    'imaging_studies': 'DATE',
    'immunizations': 'DATE',
    'medications': 'START',
    'observations': 'DATE',
    'payer_transitions': 'START_DATE',
    'procedures': 'START',
    'supplies': 'DATE',
}

SHARED_TABLES = {
    'patients': patients,
    'organizations': organizations,
    'providers': providers,
}


def split_by_date(df, date_col, cutoff=CUT_OFF_DATE):
    """Return (current, historical) using current = date >= 2020-01-01."""
    if date_col not in df.columns:
        return df.copy(), df.copy()

    # Force all timestamps into the same timezone so comparisons are safe.
    dates = pd.to_datetime(df[date_col], errors='coerce', utc=True)
    valid = dates.notna()
    current = df.loc[valid & (dates >= cutoff)].copy()
    historical = df.loc[valid & (dates < cutoff)].copy()
    return current, historical


current_tables = {}
historical_tables = {}

for name, date_col in DATE_COLUMNS.items():
    current_df, historical_df = split_by_date(globals()[name], date_col)
    current_tables[name] = current_df
    historical_tables[name] = historical_df
    print(f"{name}: current={current_df.shape}, historical={historical_df.shape}")

for name, df in SHARED_TABLES.items():
    current_tables[name] = df.copy()
    historical_tables[name] = df.copy()
    print(f"{name}: shared across both halves -> {df.shape}")

# Convenience aliases for the main tables you are most likely to use
current_encounters = current_tables['encounters']
historical_encounters = historical_tables['encounters']
current_observations = current_tables['observations']
historical_observations = historical_tables['observations']
current_conditions = current_tables['conditions']
historical_conditions = historical_tables['conditions']
current_medications = current_tables['medications']
historical_medications = historical_tables['medications']
current_procedures = current_tables['procedures']
historical_procedures = historical_tables['procedures']


allergies: current=(172, 15), historical=(2328, 15)
careplans: current=(2194, 9), historical=(5972, 9)
claims_transactions: current=(850363, 33), historical=(1126010, 33)
claims: current=(88436, 31), historical=(139651, 31)
conditions: current=(484, 11), historical=(762, 11)
devices: current=(4617, 7), historical=(7547, 7)
encounters: current=(48301, 15), historical=(77747, 15)
imaging_studies: current=(71473, 13), historical=(122396, 13)
immunizations: current=(16542, 6), historical=(15870, 6)
medications: current=(40178, 13), historical=(62224, 13)
observations: current=(664119, 9), historical=(870922, 9)
payer_transitions: current=(9963, 8), historical=(70580, 8)
procedures: current=(149779, 10), historical=(193333, 10)
supplies: current=(22417, 6), historical=(33232, 6)
patients: shared across both halves -> (2823, 28)
organizations: shared across both halves -> (3, 11)
providers: shared across both halves -> (2, 13)


In [37]:
globals()

{'__name__': '__main__',
 '__doc__': 'Automatically created module for IPython interactive environment',
 '__package__': None,
 '__loader__': None,
 '__spec__': None,
 '__builtin__': <module 'builtins' (built-in)>,
 '__builtins__': <module 'builtins' (built-in)>,
 '_ih': ['',
  "get_ipython().run_line_magic('pip', 'install numpy pandas matplotlib seaborn scikit-learn')",
  "datasets = ['allergies.csv','careplans.csv',\n            'claims_transactions.csv',\n            'claims.csv',\n            'devices.csv',\n            'encounters.csv',\n            'imaging_studies.csv',\n            'immunizations.csv',\n            'medications.csv',\n            'observations.csv',\n            'organizations.csv',\n            'patients.csv',\n            'procedures.csv',\n            'payer_transitions.csv',\n            'providers.csv',\n            'supplies.csv',]",
  'import pandas as pd\nfor dataset in datasets:\n    print(f"Loading {dataset}...")\n    df = pd.read_csv(dataset)\n    pr

In [30]:
import pandas as pd
for dataset in datasets:
    print(f"Loading {dataset}...")
    df = pd.read_csv(dataset, on_bad_lines='skip', engine='python')
    #print(f"Shape of {dataset}: {df.shape}")
    print(f"Columns in {dataset}: {df.columns.tolist()}")
    #print(f'dataset description: {df.describe()}')
    #print(f"Missing values in {dataset}:\n{df.isnull().sum()}\n")

Loading ./csv/allergies.csv...
Columns in ./csv/allergies.csv: ['START', 'STOP', 'PATIENT', 'ENCOUNTER', 'CODE', 'SYSTEM', 'DESCRIPTION', 'TYPE', 'CATEGORY', 'REACTION1', 'DESCRIPTION1', 'SEVERITY1', 'REACTION2', 'DESCRIPTION2', 'SEVERITY2']
Loading ./csv/careplans.csv...
Columns in ./csv/careplans.csv: ['Id', 'START', 'STOP', 'PATIENT', 'ENCOUNTER', 'CODE', 'DESCRIPTION', 'REASONCODE', 'REASONDESCRIPTION']
Loading ./csv/claims_transactions.csv...
Columns in ./csv/claims_transactions.csv: ['ID', 'CLAIMID', 'CHARGEID', 'PATIENTID', 'TYPE', 'AMOUNT', 'METHOD', 'FROMDATE', 'TODATE', 'PLACEOFSERVICE', 'PROCEDURECODE', 'MODIFIER1', 'MODIFIER2', 'DIAGNOSISREF1', 'DIAGNOSISREF2', 'DIAGNOSISREF3', 'DIAGNOSISREF4', 'UNITS', 'DEPARTMENTID', 'NOTES', 'UNITAMOUNT', 'TRANSFEROUTID', 'TRANSFERTYPE', 'PAYMENTS', 'ADJUSTMENTS', 'TRANSFERS', 'OUTSTANDING', 'APPOINTMENTID', 'LINENOTE', 'PATIENTINSURANCEID', 'FEESCHEDULEID', 'PROVIDERID', 'SUPERVISINGPROVIDERID']
Loading ./csv/claims.csv...
Columns in ./

# what kind of insights can we generate with all of this data?

essentially we have 3 big categories: patient demographics, clinical observations, and diagnosed medical conditions. 

we can track patient history - cuz there's multiple records for a single patient.
- in a better structured wording: the dataset includes longitudinal clinical observations across multiple
encounters, enabling the aggregation and analysis of patient-specific trends and patterns.
-Patient-related attributes describe demographic and socio-economic characteristics such as age,
gender, geographic location, and income. Clinical observation features quantify a wide range of
physiological measurements, including vital signs and laboratory test results, reflecting the
patient’s health condition at different points in time. Condition-related data capture diagnosed
diseases using standardized medical codes and descriptive labels, enabling the identification of
clinically relevant outcomes.




In [36]:
encounters.head(20)

,Id,START,STOP,PATIENT,ORGANIZATION,PROVIDER,PAYER,ENCOUNTERCLASS,CODE,DESCRIPTION,BASE_ENCOUNTER_COST,TOTAL_CLAIM_COST,PAYER_COVERAGE,REASONCODE,REASONDESCRIPTION
0,0f238ca9-106f-4cd6-5a5a-88f8c5f174fd,2001-01-10T10:39:55Z,2001-01-10T10:54:55Z,0f238ca9-106f-4cd6-d8d1-701e70c84cb6,3f12ebb4-e03c-3453-88d2-4fc9682383df,57296f8e-d827-357f-b313-94cae3b30953,df166300-5a78-3502-a46a-832842197811,wellness,410620009,Well child visit (procedure),136.80,347.38,197.38,NaN,NaN
1,0f238ca9-106f-4cd6-a21d-04f386786bb5,2001-02-14T10:39:55Z,2001-02-14T10:54:55Z,0f238ca9-106f-4cd6-d8d1-701e70c84cb6,3f12ebb4-e03c-3453-88d2-4fc9682383df,57296f8e-d827-357f-b313-94cae3b30953,df166300-5a78-3502-a46a-832842197811,wellness,410620009,Well child visit (procedure),136.80,931.62,881.62,NaN,NaN
2,0f238ca9-106f-4cd6-60bc-ff708b49785f,2001-04-18T10:39:55Z,2001-04-18T10:54:55Z,0f238ca9-106f-4cd6-d8d1-701e70c84cb6,3f12ebb4-e03c-3453-88d2-4fc9682383df,57296f8e-d827-357f-b313-94cae3b30953,df166300-5a78-3502-a46a-832842197811,wellness,410620009,Well child visit (procedure),136.80,1238.35,1188.35,NaN,NaN
3,0f238ca9-106f-4cd6-01a7-b4a232d5f15c,2001-04-25T10:39:55Z,2001-04-25T13:29:27Z,0f238ca9-106f-4cd6-d8d1-701e70c84cb6,d140ed3f-2696-3e78-bd77-1c959ec4fc4d,3d979e03-03df-3971-904b-cb8fe69edf05,df166300-5a78-3502-a46a-832842197811,emergency,50849002,Emergency room admission (procedure),146.18,4407.08,4357.08,128613002.0,Seizure disorder (disorder)
4,0f238ca9-106f-4cd6-924d-3b688876e69b,2001-06-20T10:39:55Z,2001-06-20T10:54:55Z,0f238ca9-106f-4cd6-d8d1-701e70c84cb6,3f12ebb4-e03c-3453-88d2-4fc9682383df,57296f8e-d827-357f-b313-94cae3b30953,df166300-5a78-3502-a46a-832842197811,wellness,410620009,Well child visit (procedure),136.80,1407.60,1357.60,NaN,NaN
5,0f238ca9-106f-4cd6-9b89-b082efd0613d,2001-08-29T10:39:55Z,2001-09-21T13:29:27Z,0f238ca9-106f-4cd6-d8d1-701e70c84cb6,a418cf76-2ee3-3736-a752-29ee33e86015,f0ed7816-e346-34a1-bd9c-3309eb9a756b,df166300-5a78-3502-a46a-832842197811,hospice,305336008,Admission to hospice (procedure),137.53,11353.93,11303.93,NaN,NaN
6,0f238ca9-106f-4cd6-7bb2-1eb6557734c4,2001-09-26T10:39:55Z,2001-09-26T10:54:55Z,0f238ca9-106f-4cd6-d8d1-701e70c84cb6,3f12ebb4-e03c-3453-88d2-4fc9682383df,57296f8e-d827-357f-b313-94cae3b30953,df166300-5a78-3502-a46a-832842197811,wellness,308646001,Death Certification,136.80,211.38,161.38,95281009.0,Sudden cardiac death (disorder)
7,a35facb9-a2fd-363c-2bdf-77ec22718268,1972-06-07T08:30:20Z,1972-06-07T08:45:20Z,a35facb9-a2fd-363c-dc7f-5f32ed850c45,c6b019eb-28ec-36f6-abf3-bcc4d1d58966,cfb8e36a-33e4-3c70-9099-5ceb308a747c,df166300-5a78-3502-a46a-832842197811,wellness,410620009,Well child visit (procedure),136.80,272.80,122.80,NaN,NaN
8,a35facb9-a2fd-363c-55df-fc73bda41499,1973-09-05T08:30:20Z,1973-09-05T08:45:20Z,a35facb9-a2fd-363c-dc7f-5f32ed850c45,c6b019eb-28ec-36f6-abf3-bcc4d1d58966,cfb8e36a-33e4-3c70-9099-5ceb308a747c,df166300-5a78-3502-a46a-832842197811,wellness,410620009,Well child visit (procedure),136.80,136.80,86.80,NaN,NaN
9,a35facb9-a2fd-363c-cb53-9ab814a35bc5,1973-10-21T17:30:20Z,1973-10-21T17:45:20Z,a35facb9-a2fd-363c-dc7f-5f32ed850c45,b6eeaaf7-1683-3bcb-b6ee-81ce304636ef,62989d7d-9ce7-34da-8a6a-eae35f320fee,df166300-5a78-3502-a46a-832842197811,ambulatory,185345009,Encounter for symptom (procedure),85.55,85.55,35.55,444814009.0,Viral sinusitis (disorder)


# EHR Table Merging & Temporal Split

**Strategy:**
- `encounters` is the anchor table — it has a timestamp (`START`) and links every clinical event back to a patient.
- Patient demographics from `patients` are merged directly.
- Encounter-keyed tables (procedures, medications, observations, imaging, immunizations, devices, supplies) are **aggregated per encounter** before merging, to keep the dataset at encounter granularity.
- Patient-keyed tables (allergies, careplans, payer_transitions) are **aggregated per patient** and merged via the `PATIENT` column.
- **Temporal split** uses the median `START` date as the cutoff → Dataset 1 (Historical) and Dataset 2 (Current).

In [28]:
allergies[allergies['START'] == allergies['START'].min()]

,START,STOP,PATIENT,ENCOUNTER,CODE,SYSTEM,DESCRIPTION,TYPE,CATEGORY,REACTION1,DESCRIPTION1,SEVERITY1,REACTION2,DESCRIPTION2,SEVERITY2
1384,1918-11-19,NaN,ad5c5147-cf71-0511-b511-f17030a21510,ad5c5147-cf71-0511-c849-c5aa5193ae3d,609328004,SNOMED-CT,Allergic disposition (finding),allergy,environment,NaN,NaN,NaN,NaN,NaN,NaN
1385,1918-11-19,NaN,ad5c5147-cf71-0511-b511-f17030a21510,ad5c5147-cf71-0511-c849-c5aa5193ae3d,102263004,SNOMED-CT,Eggs (edible) (substance),allergy,food,247472004.0,Wheal (finding),MODERATE,NaN,NaN,NaN


In [29]:
careplans.head()

,Id,START,STOP,PATIENT,ENCOUNTER,CODE,DESCRIPTION,REASONCODE,REASONDESCRIPTION
0,a35facb9-a2fd-363c-70c6-9490ec3895dd,1975-10-17,NaN,a35facb9-a2fd-363c-dc7f-5f32ed850c45,a35facb9-a2fd-363c-6572-3a922bd53761,384758001,Self-care interventions (procedure),NaN,NaN
1,e3e20559-00f1-a5b1-a5bd-7ac29d7e5fd9,2017-03-20,NaN,e3e20559-00f1-a5b1-0e26-4145235cccbf,e3e20559-00f1-a5b1-6592-49985bc9750c,384758001,Self-care interventions (procedure),NaN,NaN
2,e3e20559-00f1-a5b1-1c20-d429e96f6dc0,2022-05-06,2022-06-06,e3e20559-00f1-a5b1-0e26-4145235cccbf,e3e20559-00f1-a5b1-4a1f-0d55637bea8b,773513001,Physiotherapy care plan (record artifact),44465007.0,Sprain of ankle (disorder)
3,e3e20559-00f1-a5b1-fb99-f7aae943d66a,2025-08-23,2025-08-31,e3e20559-00f1-a5b1-0e26-4145235cccbf,e3e20559-00f1-a5b1-f813-30f9ebb3c68b,53950000,Respiratory therapy (procedure),NaN,NaN
4,90f0bc22-78e9-3b0c-cc14-34e55641b1c8,2010-09-09,NaN,90f0bc22-78e9-3b0c-0479-e2d5091cba18,90f0bc22-78e9-3b0c-6bfe-64e2df09aae3,718361005,Weight management program (regime/therapy),NaN,NaN


# so we have effectively 2 approaches we can take: 
Disease Onset Prediction (Clinical): Can you predict if a patient will be diagnosed with a specific chronic condition (e.g., Type 2 Diabetes or Hypertension in careplans.csv or claims.csv) 6 to 12 months before the diagnosis? You would use the trend of their vital signs and lab results (observations.csv) and demographic risk factors (patients.csv) as your inputs.

Hospital Readmission Risk (Operational): Look at the encounters.csv. If a patient is discharged from an inpatient encounter, what is the probability they will be admitted again within 30 days? This is a massive real-world metric for hospitals.


# Exploratory Data Analysis

This EDA is structured around **six themes**, applied consistently to both the **Current (≥2021)** and **Historical (<2021)** splits:

| # | Theme | Key Questions |
|---|-------|---------------|
| 1 | **Dataset Overview** | How are records distributed across tables? Where are the data gaps? |
| 2 | **Patient Demographics** | Who are the patients? Age, gender, race, income |
| 3 | **Temporal Trends** | How do encounter volumes vary by year and month? |
| 4 | **Encounter Class & Cost** | What visit types occur? How do costs compare? |
| 5 | **Clinical Patterns** | Which conditions, medications, and vitals define this population? |
| 6 | **Financial Analysis** | How well does payer coverage track actual claim costs? |

> **Color convention throughout**: `steelblue` = Historical, `coral` = Current

In [ ]:
# ── EDA Global Style Setup ────────────────────────────────────────────────
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns

# Consistent visual theme for all EDA plots
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#f8f9fa',
    'figure.dpi':        110,
})
print("EDA style configured — steelblue = Historical, coral = Current throughout.")

In [ ]:
# ── Dataset Overview: Row Counts & Missing Values ─────────────────────────
# Gives a bird's-eye view of how records are distributed across tables and
# where gaps in the data could affect downstream feature engineering.

TRACKED_TABLES = [
    'encounters', 'conditions', 'medications', 'procedures',
    'observations', 'allergies', 'careplans', 'imaging_studies',
    'immunizations', 'devices', 'supplies', 'claims',
]

# Summary DataFrame: historical vs current row counts
summary_df = pd.DataFrame({
    'Table':      TRACKED_TABLES,
    'Historical': [historical_tables[t].shape[0] for t in TRACKED_TABLES],
    'Current':    [current_tables[t].shape[0]    for t in TRACKED_TABLES],
})
summary_df['Total']     = summary_df['Historical'] + summary_df['Current']
summary_df['% Current'] = (summary_df['Current'] / summary_df['Total'] * 100).round(1)
print(summary_df.to_string(index=False))

# Grouped bar chart: historical vs current record counts
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(TRACKED_TABLES))
w = 0.4
ax.bar(x - w/2, summary_df['Historical'], w,
       label='Historical (<2021)', color='steelblue')
ax.bar(x + w/2, summary_df['Current'],    w,
       label='Current (≥2021)',    color='coral')
ax.set_xticks(x)
ax.set_xticklabels(TRACKED_TABLES, rotation=40, ha='right')
ax.set_title('Record Counts: Historical vs Current Split Across All Tables')
ax.set_ylabel('Number of Records')
ax.legend()
plt.tight_layout()
plt.show()

# Missing-value profile for encounters (the central join table)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (label, df) in zip(axes, [
    ('Current Encounters',    current_encounters),
    ('Historical Encounters', historical_encounters),
]):
    null_pct = df.isnull().mean().sort_values(ascending=False)
    null_pct = null_pct[null_pct > 0]   # only show columns with any nulls
    null_pct.plot(kind='bar', ax=ax, color='salmon', edgecolor='white')
    ax.set_title(f'Missing Value Rate — {label}')
    ax.set_ylabel('Fraction of Rows Missing')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## Dataset Overview — Insights

- **Size asymmetry**: Historical data (pre-2021) contains roughly 1.5–2× more records than the current window across most tables. This is expected — the historical split accumulates over decades while "current" spans only ~4 years (2021–2025). Any model trained on historical data and evaluated on current data must account for this volume difference.
- **Largest tables**: `claims_transactions` (~2M rows), `observations` (~1.5M rows), and `procedures` (~340k rows) dominate. Joins involving these tables should be carefully scoped to avoid memory blowup.
- **Missing values in encounters**: `REASONCODE` and `REASONDESCRIPTION` are the main sources of missingness — by clinical design, not every encounter has a coded presenting complaint. `PAYER_COVERAGE` and cost columns are almost always present. When using reason codes as model features, an explicit "no reason coded" binary indicator is more informative than imputation.

In [ ]:
# ── Patient Demographics ───────────────────────────────────────────────────
# Patients table is shared across both periods, so we analyse it once.

pts = patients.copy()
pts['BIRTHDATE'] = pd.to_datetime(pts['BIRTHDATE'], errors='coerce')
pts['DEATHDATE'] = pd.to_datetime(pts['DEATHDATE'], errors='coerce')

# For living patients age = today − birthdate; for deceased = death − birth
ANALYSIS_DATE = pd.Timestamp('2026-04-20')
pts['AGE'] = np.where(
    pts['DEATHDATE'].isna(),
    (ANALYSIS_DATE - pts['BIRTHDATE']).dt.days / 365.25,
    (pts['DEATHDATE']  - pts['BIRTHDATE']).dt.days / 365.25,
)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Age histogram with median marker
ax = axes[0, 0]
pts['AGE'].clip(0, 100).hist(bins=30, ax=ax, color='steelblue', edgecolor='white')
med_age = pts['AGE'].clip(0, 100).median()
ax.axvline(med_age, color='red', linestyle='--', label=f'Median: {med_age:.1f} yrs')
ax.set_title('Age Distribution')
ax.set_xlabel('Age (years)')
ax.set_ylabel('Count')
ax.legend()

# 2. Gender breakdown as a pie chart
ax = axes[0, 1]
gc = pts['GENDER'].value_counts()
ax.pie(gc, labels=gc.index, autopct='%1.1f%%',
       colors=['#4C9BE8', '#FF8C8C', '#A8E6A3'],
       startangle=90, wedgeprops=dict(edgecolor='white', linewidth=1.5))
ax.set_title('Gender Distribution')

# 3. Race breakdown as a horizontal bar chart
ax = axes[1, 0]
rc = pts['RACE'].value_counts()
rc.sort_values().plot(kind='barh', ax=ax,
                      color=sns.color_palette('muted', len(rc)), edgecolor='white')
ax.set_title('Race Distribution')
ax.set_xlabel('Count')

# 4. Income distribution clipped to 99th percentile
ax = axes[1, 1]
income = pd.to_numeric(pts['INCOME'], errors='coerce').dropna()
income.clip(0, income.quantile(0.99)).hist(bins=40, ax=ax, color='teal', edgecolor='white')
ax.set_title('Annual Income Distribution')
ax.set_xlabel('Income ($)')
ax.set_ylabel('Count')

plt.suptitle('Patient Demographics Overview', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# --- Healthcare expenses vs payer coverage scatter ---
# Reveals which patients bear a disproportionate financial burden
fig, ax = plt.subplots(figsize=(8, 5))
expenses = pd.to_numeric(pts['HEALTHCARE_EXPENSES'], errors='coerce')
coverage  = pd.to_numeric(pts['HEALTHCARE_COVERAGE'],  errors='coerce')
exp_clip  = expenses.clip(0, expenses.quantile(0.99))
cov_clip  = coverage.clip(0, coverage.quantile(0.99))
ax.scatter(exp_clip, cov_clip, alpha=0.3, s=10, color='steelblue')
ax.set_title('Healthcare Expenses vs Payer Coverage per Patient')
ax.set_xlabel('Total Healthcare Expenses ($)')
ax.set_ylabel('Total Payer Coverage ($)')
plt.tight_layout()
plt.show()

## Patient Demographics — Insights

- **Age distribution**: The patient population spans all life stages, with a broad peak in the 40–70 age range. The distribution has a secondary hump for paediatric patients, reflecting well-child visits. The high proportion of older adults is consistent with a system dominated by chronic-disease management.
- **Gender**: Near-even split (~50/50), so gender imbalance is unlikely to confound downstream models. However, differential care patterns by gender (e.g., cardiovascular disease presentation differences) are still worth investigating.
- **Race**: The racial composition reflects a US-like synthetic population. White patients form the plurality, with meaningful representation of Black, Hispanic/Latino, and Asian patients. Building equitable models requires sub-group performance checks — aggregated AUC can mask poor performance for minority groups.
- **Income**: Right-skewed with most patients earning below $100k annually. Socioeconomic status is a well-established social determinant of health; low income correlates with delayed care-seeking, poor medication adherence, and higher readmission rates. Income should be treated as a first-class feature, not just a demographic covariate.
- **Expenses vs Coverage scatter**: A linear relationship confirms that payer coverage scales with total expenses, but the variance is wide — some patients have very high expenses with low coverage, identifying the highest-risk financial outliers.

In [ ]:
# ── Temporal Distribution of Encounters ──────────────────────────────────
# Parsing encounter start dates to examine yearly trends and monthly seasonality.

curr_enc_t = current_encounters.copy()
hist_enc_t = historical_encounters.copy()

curr_enc_t['START_DT'] = pd.to_datetime(curr_enc_t['START'], utc=True, errors='coerce')
hist_enc_t['START_DT'] = pd.to_datetime(hist_enc_t['START'], utc=True, errors='coerce')

curr_enc_t['YEAR']  = curr_enc_t['START_DT'].dt.year
hist_enc_t['YEAR']  = hist_enc_t['START_DT'].dt.year
curr_enc_t['MONTH'] = curr_enc_t['START_DT'].dt.month

# --- Yearly encounter volume ---
curr_yr = curr_enc_t['YEAR'].value_counts().sort_index()
hist_yr = hist_enc_t['YEAR'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
curr_yr.plot(kind='bar', ax=axes[0], color='coral', edgecolor='white')
axes[0].set_title('Encounters per Year — Current Period (≥2021)')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Encounter Count')
axes[0].tick_params(axis='x', rotation=30)

# Line chart for the long historical range
hist_yr.plot(kind='line', ax=axes[1], color='steelblue',
             marker='o', markersize=2.5, linewidth=1.5)
axes[1].set_title('Encounters per Year — Historical Period (<2021)')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Encounter Count')
plt.suptitle('Temporal Encounter Volume', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# --- Monthly seasonality across all years in the current period ---
monthly = curr_enc_t.groupby('MONTH').size()
MONTH_LABELS = ['Jan','Feb','Mar','Apr','May','Jun',
                'Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(10, 4))
monthly.plot(kind='bar', ax=ax, color='mediumseagreen', edgecolor='white')
ax.set_xticks(range(12))
ax.set_xticklabels(MONTH_LABELS, rotation=30)
ax.set_title('Monthly Encounter Seasonality — Current Period (all years combined)')
ax.set_ylabel('Total Encounter Count')
plt.tight_layout()
plt.show()

## Temporal Distribution of Encounters — Insights

- **Historical period (pre-2021)**: Encounter volume grows steadily over the decades, reflecting both population growth and longer patient histories accumulating in the record. The sharp drop after ~2020 is artificial — those records are assigned to the current split by design.
- **Current period (2021+)**: Annual encounter volumes are relatively consistent across years, with a slight upward trend. Any dip in 2021–2022 could reflect modelled pandemic-era access disruption in the synthetic generator.
- **Monthly seasonality**: A mild but interpretable seasonal pattern emerges — peaks in winter months (Jan–Mar) driven by respiratory illness season, and lower activity in summer. This seasonality is consistent with real clinical data and should be accounted for in any time-series forecasting or readmission model that uses encounter frequency as a feature.

In [ ]:
# ── Encounter Class & Cost Analysis ──────────────────────────────────────

# --- Encounter class counts: side-by-side for current vs historical ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (label, df) in zip(axes, [('Current (≥2021)', current_encounters),
                                    ('Historical (<2021)', historical_encounters)]):
    counts = df['ENCOUNTERCLASS'].value_counts()
    counts.plot(kind='bar', ax=ax,
                color=sns.color_palette('Set2', len(counts)), edgecolor='white')
    ax.set_title(f'Encounter Class — {label}')
    ax.set_xlabel('Encounter Class')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=30)
plt.suptitle('Encounter Class Distribution: Current vs Historical',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# --- Log(total claim cost) by encounter class — boxplot reveals spread ---
combined_enc_cost = pd.concat([
    current_encounters.assign(Period='Current'),
    historical_encounters.assign(Period='Historical')
], ignore_index=True)
combined_enc_cost['TOTAL_CLAIM_COST'] = pd.to_numeric(
    combined_enc_cost['TOTAL_CLAIM_COST'], errors='coerce')
combined_enc_cost['PAYER_COVERAGE'] = pd.to_numeric(
    combined_enc_cost['PAYER_COVERAGE'], errors='coerce')
combined_enc_cost['LOG_COST'] = np.log1p(combined_enc_cost['TOTAL_CLAIM_COST'])

fig, ax = plt.subplots(figsize=(13, 5))
sns.boxplot(
    data=combined_enc_cost, x='ENCOUNTERCLASS', y='LOG_COST',
    hue='Period', palette={'Current': 'coral', 'Historical': 'steelblue'},
    ax=ax, flierprops=dict(marker='o', markersize=2, alpha=0.3)
)
ax.set_title('log(1 + Total Claim Cost) by Encounter Class — Current vs Historical')
ax.set_xlabel('Encounter Class')
ax.set_ylabel('log(1 + Total Claim Cost)')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

# --- Out-of-pocket cost: what patients pay after insurance ---
combined_enc_cost['OOP'] = (
    combined_enc_cost['TOTAL_CLAIM_COST'] - combined_enc_cost['PAYER_COVERAGE']
).clip(lower=0)

fig, ax = plt.subplots(figsize=(10, 4))
for period, color in [('Historical', 'steelblue'), ('Current', 'coral')]:
    oop = combined_enc_cost.loc[combined_enc_cost['Period'] == period, 'OOP'].dropna()
    oop = oop[oop < oop.quantile(0.99)]  # suppress extreme upper tail for legibility
    ax.hist(oop, bins=60, alpha=0.65, color=color, label=period, density=True)
ax.set_title('Out-of-Pocket Cost Distribution  (Total Claim − Payer Coverage)')
ax.set_xlabel('Out-of-Pocket Cost ($)')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()

## Encounter Class & Cost Analysis — Insights

- **Encounter class distribution**: Wellness and ambulatory encounters dominate both periods — expected for a primary-care-centric dataset. Emergency visits are a small but high-cost minority, consistent with real-world patterns where ED admissions are episodic and expensive.
- **Claim cost by encounter class (log scale)**: Inpatient and hospice encounters have the widest spread and highest medians. Wellness visits cluster tightly at low cost; emergency encounters show a long right tail driven by complex, resource-intensive workups. The log transformation reveals cost structure that is hidden in raw histograms.
- **Period comparison**: Current encounters trend slightly higher in median cost across most classes, consistent with healthcare cost inflation. The shape of each distribution is stable, so period is primarily a scale shift rather than a structural change.
- **Out-of-pocket (OOP) cost**: The distribution is right-skewed with a spike near $0 (fully covered patients) and a long tail of patients bearing significant costs. This financial burden is concentrated in uninsured / underinsured patients who present predominantly via emergency encounters — a group at elevated readmission risk.

In [ ]:
# ── Condition / Diagnosis Analysis ────────────────────────────────────────

# --- Top 15 conditions by frequency in each temporal window ---
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, (label, df) in zip(axes, [('Current (≥2021)', current_conditions),
                                    ('Historical (<2021)', historical_conditions)]):
    top = df['DESCRIPTION'].value_counts().head(15).sort_values()
    top.plot(kind='barh', ax=ax, color='mediumpurple', edgecolor='white')
    ax.set_title(f'Top 15 Conditions — {label}')
    ax.set_xlabel('Count')
plt.suptitle('Most Frequent Diagnoses: Current vs Historical',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# --- Conditions per patient: measures clinical complexity ---
cond_per_pt_curr = current_conditions.groupby('PATIENT').size()
cond_per_pt_hist = historical_conditions.groupby('PATIENT').size()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(cond_per_pt_hist, bins=30, alpha=0.65, color='steelblue',
        label=f'Historical (mean={cond_per_pt_hist.mean():.1f})', density=True)
ax.hist(cond_per_pt_curr, bins=30, alpha=0.65, color='coral',
        label=f'Current (mean={cond_per_pt_curr.mean():.1f})',    density=True)
ax.set_title('Conditions per Patient — Clinical Complexity Distribution')
ax.set_xlabel('Number of Distinct Conditions')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()

## Conditions Analysis — Insights

- **Top conditions** in both periods are dominated by common chronic diseases — hypertension, diabetes, hyperlipidaemia, and anxiety/depression. This mirrors the real-world US epidemiological burden of non-communicable diseases and validates the fidelity of the synthetic data.
- **Period-specific patterns**: Some conditions appear more prominently in the current window (e.g., COVID-19 adjacent respiratory diagnoses, newer ICD-10 codes), while older infection classifications dominate historically. This reflects both genuine disease incidence changes and evolving coding practices.
- **Conditions per patient**: The distribution is heavily right-skewed — most patients carry 1–3 active diagnoses, but a clinically critical long tail has 10+ conditions. This multi-morbidity tail is exactly the high-risk cohort most relevant for predictive modelling (readmission risk, escalation of care, medication adherence). Any model should ensure this group is well-represented in the training set.

In [ ]:
# ── Medications Analysis ───────────────────────────────────────────────────

# --- Top 15 medications by frequency ---
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, (label, df) in zip(axes, [('Current (≥2021)', current_medications),
                                    ('Historical (<2021)', historical_medications)]):
    top = df['DESCRIPTION'].value_counts().head(15).sort_values()
    top.plot(kind='barh', ax=ax, color='darkorange', edgecolor='white')
    ax.set_title(f'Top 15 Medications by Frequency — {label}')
    ax.set_xlabel('Number of Prescriptions')
plt.suptitle('Medication Frequency Comparison: Current vs Historical',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# --- Total medication cost distribution (log-transformed) ---
curr_cost = pd.to_numeric(current_medications['TOTALCOST'],    errors='coerce').dropna()
hist_cost = pd.to_numeric(historical_medications['TOTALCOST'], errors='coerce').dropna()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(np.log1p(hist_cost), bins=50, alpha=0.65, color='steelblue',
        label='Historical', density=True)
ax.hist(np.log1p(curr_cost), bins=50, alpha=0.65, color='coral',
        label='Current',    density=True)
ax.set_title('Total Medication Cost Distribution (log scale)')
ax.set_xlabel('log(1 + Total Cost $)')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()

# --- Medications per patient: how many prescriptions does each patient receive? ---
meds_per_pt_curr = current_medications.groupby('PATIENT').size()
meds_per_pt_hist = historical_medications.groupby('PATIENT').size()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(meds_per_pt_hist, bins=40, alpha=0.65, color='steelblue',
        label=f'Historical (mean={meds_per_pt_hist.mean():.1f})', density=True)
ax.hist(meds_per_pt_curr, bins=40, alpha=0.65, color='coral',
        label=f'Current (mean={meds_per_pt_curr.mean():.1f})',    density=True)
ax.set_title('Number of Medication Prescriptions per Patient')
ax.set_xlabel('Prescriptions per Patient')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()

## Medications Analysis — Insights

- **Top medications** shift meaningfully between periods. Historically, older drug formulations (e.g., first-generation antihistamines, older antibiotics) dominate. The current period features more targeted therapies and newer formulations, reflecting evolving clinical guidelines and formulary changes.
- **Medication cost** distributions are highly right-skewed on the raw scale (near-symmetric on log scale). A small fraction of prescriptions — typically biologics or specialty drugs — account for a disproportionately large share of total pharmaceutical spending. This 80/20 pattern is well-established in pharmacoeconomics.
- **Current costs skew higher**: The current distribution is shifted slightly right of historical on the log scale, capturing both healthcare cost inflation and the introduction of more expensive drug classes over time.
- **Medications per patient**: Most patients receive 1–5 medication courses in a given period, but a long tail of 10+ prescriptions per patient captures polypharmacy — a significant risk factor for adverse drug events and non-adherence. This tail is the highest-risk segment for readmission models.

In [ ]:
# ── Key Vital Signs: Current vs Historical ────────────────────────────────
# We extract 6 physiological measurements from the observations table and
# overlay their distributions across both temporal windows.

VITAL_MAP = {
    'Body mass index (BMI) [Ratio]': 'BMI (kg/m²)',
    'Systolic Blood Pressure':       'Systolic BP (mmHg)',
    'Diastolic Blood Pressure':      'Diastolic BP (mmHg)',
    'Heart rate':                    'Heart Rate (bpm)',
    'Body Weight':                   'Body Weight (kg)',
    'Respiratory rate':              'Respiratory Rate (br/min)',
}

def extract_vital(obs_df, description):
    """Return a numeric series for the given DESCRIPTION label in observations."""
    return (
        pd.to_numeric(
            obs_df.loc[obs_df['DESCRIPTION'] == description, 'VALUE'],
            errors='coerce'
        ).dropna()
    )

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, (desc, label) in zip(axes.flatten(), VITAL_MAP.items()):
    curr_v = extract_vital(current_observations,   desc)
    hist_v = extract_vital(historical_observations, desc)

    # Clip to [1st, 99th] percentile shared range to suppress extreme outliers
    lo = min(curr_v.quantile(0.01), hist_v.quantile(0.01))
    hi = max(curr_v.quantile(0.99), hist_v.quantile(0.99))

    ax.hist(hist_v.clip(lo, hi), bins=40, alpha=0.65, color='steelblue',
            label='Historical', density=True)
    ax.hist(curr_v.clip(lo, hi), bins=40, alpha=0.65, color='coral',
            label='Current',    density=True)
    ax.set_title(label)
    ax.set_xlabel(label)
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.suptitle('Key Vital Signs — Current vs Historical', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Vital Signs — Insights

- **BMI**: Distribution is right-skewed with a peak in the 25–30 range (overweight threshold), consistent with US population trends. A long right tail indicates a meaningful subset of patients with obesity — a key comorbidity driver for diabetes, hypertension, and sleep apnoea.
- **Systolic & Diastolic BP**: Both are approximately normal. Systolic peaks near 120 mmHg; diastolic near 80 mmHg — reflecting a mixed healthy and pre-hypertensive population. The near-identical current vs historical shapes suggest the patient pool's cardiovascular profile is stable across time.
- **Heart Rate**: Approximately normally distributed around 70–80 bpm. Outliers at both extremes are typically captured during acute illness or post-exercise encounters and do not indicate data error.
- **Body Weight**: Right-skewed (median ~75–85 kg) in line with a general adult US population. The current period shows a slight rightward shift, hinting at a modest secular trend toward higher body weight over the recorded decades.
- **Respiratory Rate**: Tightly clustered around 12–16 br/min (normal range), with a tail toward higher values corresponding to respiratory-distress encounters. Current and historical distributions are virtually identical — this vital is less sensitive to secular trends than BMI or weight.

In [ ]:
# ── Financial & Claims Analysis ────────────────────────────────────────────
# We look at (1) primary-insurance outstanding balances, (2) payer coverage
# ratio per encounter, and (3) mean coverage ratio by encounter class.

curr_claims = current_tables['claims'].copy()
hist_claims = historical_tables['claims'].copy()

# --- 1. Outstanding balance distribution ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (label, df) in zip(axes, [('Current (≥2021)', curr_claims),
                                    ('Historical (<2021)', hist_claims)]):
    outs = pd.to_numeric(df['OUTSTANDING1'], errors='coerce').dropna()
    # Clip to 1st–99th percentile for visual clarity
    outs = outs.clip(outs.quantile(0.01), outs.quantile(0.99))
    ax.hist(outs, bins=50, color='teal', edgecolor='white')
    ax.axvline(outs.median(), color='red', linestyle='--',
               label=f'Median: ${outs.median():.2f}')
    ax.set_title(f'Primary Insurance Outstanding — {label}')
    ax.set_xlabel('Outstanding Amount ($)')
    ax.set_ylabel('Count')
    ax.legend()
plt.suptitle('Outstanding Claim Balances (Primary Insurance)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# --- 2. Payer coverage ratio per encounter ---
combined_enc_fin = pd.concat([
    current_encounters.assign(Period='Current'),
    historical_encounters.assign(Period='Historical')
], ignore_index=True)

combined_enc_fin['TOTAL_CLAIM_COST'] = pd.to_numeric(combined_enc_fin['TOTAL_CLAIM_COST'], errors='coerce')
combined_enc_fin['PAYER_COVERAGE']   = pd.to_numeric(combined_enc_fin['PAYER_COVERAGE'],   errors='coerce')
# Coverage ratio: what fraction of total cost does the payer absorb?
combined_enc_fin['COVERAGE_RATIO'] = (
    combined_enc_fin['PAYER_COVERAGE'] /
    combined_enc_fin['TOTAL_CLAIM_COST'].replace(0, np.nan)
).clip(0, 1)

fig, ax = plt.subplots(figsize=(10, 4))
for period, color in [('Historical', 'steelblue'), ('Current', 'coral')]:
    vals = combined_enc_fin.loc[combined_enc_fin['Period'] == period, 'COVERAGE_RATIO'].dropna()
    ax.hist(vals, bins=50, alpha=0.65, color=color, label=period, density=True)
ax.set_title('Payer Coverage Ratio  (Payer Coverage ÷ Total Claim Cost)')
ax.set_xlabel('Coverage Ratio  (0 = no coverage, 1 = fully covered)')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()

# --- 3. Mean coverage ratio broken down by encounter class ---
coverage_cls = (
    combined_enc_fin
    .groupby(['ENCOUNTERCLASS', 'Period'])['COVERAGE_RATIO']
    .mean()
    .unstack('Period')
    .dropna()
)
fig, ax = plt.subplots(figsize=(10, 4))
coverage_cls.plot(kind='bar', ax=ax,
                  color={'Historical': 'steelblue', 'Current': 'coral'},
                  edgecolor='white')
ax.set_title('Mean Payer Coverage Ratio by Encounter Class')
ax.set_ylabel('Mean Coverage Ratio')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## Financial Analysis — Insights

- **Outstanding amounts** are skewed right in both periods, indicating most claims are fully resolved with a long tail of large balances still being processed. The current period shows slightly higher median outstanding amounts, consistent with more recent claims not yet adjudicated.
- **Payer coverage ratio** distribution is strongly bimodal: a large spike near **1.0** (fully covered) and another near **0.0** (self-pay / no coverage). This reflects the binary nature of insurance — patients either have coverage or pay entirely out-of-pocket.
- **Coverage ratio by encounter class**: Wellness and ambulatory visits tend to have the highest coverage ratios, while emergency and inpatient encounters show more variance. Some patients clearly lack emergency insurance, creating a financial vulnerability that correlates with delayed care-seeking and worse outcomes.
- The historical period shows consistently higher coverage for certain encounter classes, which may reflect changing insurance landscapes — rising deductibles and narrower networks — over the decades captured in the dataset.

# Disease Onset Prediction — Pre-Modelling Analyses

The four analyses below directly answer the question: **"Is this data good enough to build a disease onset model, and how should features be engineered?"**

| Analysis | Question Answered | Output |
|---|---|---|
| **1 — Vital Trajectory** | Do BMI/BP trend differently before onset? | Confirms trajectory features are needed (not just last value) |
| **2 — Utilization Spikes** | Do patients visit more before diagnosis? | Quantifies the help-seeking signal for encounter-count features |
| **3 — Pre-Condition Co-occurrence** | What diagnoses precede onset? | Builds the binary comorbidity feature set |
| **4 — Observation Missingness** | Which vitals can we rely on? | Defines the safe primary feature set vs optional features |

> **Target diseases**: `Any Diabetes (including Prediabetes)` and `Essential Hypertension`  
> **Positive/Negative convention**: `coral` = positive (disease eventually diagnosed), `steelblue` = negative controls  
> **Date note**: `conditions.csv` uses DD-MM-YYYY format; `dayfirst=True` is applied throughout these cells.

In [ ]:
# ── Analysis 1: Vital Sign Trajectories Before Disease Onset ──────────────
# We compare the average vital trajectory (BMI for Diabetes, Systolic BP for
# Hypertension) in the 3 years before diagnosis (positive patients) vs a
# matched 3-year window for patients who never received that diagnosis.
#
# NOTE: conditions.csv uses DD-MM-YYYY format; dayfirst=True is critical.

# Parse full (unsplit) tables — needed for longitudinal trajectory spanning decades
cond_traj = conditions.copy()
obs_traj  = observations.copy()

cond_traj['START_DT'] = pd.to_datetime(cond_traj['START'], dayfirst=True, errors='coerce')
obs_traj['DATE_DT']   = (
    pd.to_datetime(obs_traj['DATE'], utc=True, errors='coerce').dt.tz_convert(None)
)

LOOKBACK_MONTHS = 36   # 3-year trajectory window

TARGETS_TRAJ = [
    ('diabet',    'Body mass index (BMI) [Ratio]', 'BMI (kg/m²)',      'Diabetes'),
    ('hypertens', 'Systolic Blood Pressure',        'Systolic BP (mmHg)', 'Hypertension'),
]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, (kw, vital_desc, vital_label, disease_name) in zip(axes, TARGETS_TRAJ):
    # ── Positive patients: earliest onset date ────────────────────────────
    first_onset = (
        cond_traj[cond_traj['DESCRIPTION'].str.contains(kw, case=False, na=False)]
        .groupby('PATIENT')['START_DT'].min()
        .rename('ONSET_DT').reset_index()
    )
    pos_set = set(first_onset['PATIENT'])

    # Extract the target vital for all patients
    vital_obs = obs_traj[obs_traj['DESCRIPTION'] == vital_desc].copy()
    vital_obs['VAL'] = pd.to_numeric(vital_obs['VALUE'], errors='coerce')

    # ── Positive: join onset date, compute MONTH_BIN relative to onset ────
    pos_v = vital_obs[vital_obs['PATIENT'].isin(pos_set)].merge(first_onset, on='PATIENT')
    pos_v['DAYS_BEFORE'] = (pos_v['ONSET_DT'] - pos_v['DATE_DT']).dt.days
    pos_v = pos_v[(pos_v['DAYS_BEFORE'] >= 0) & (pos_v['DAYS_BEFORE'] <= LOOKBACK_MONTHS * 30.44)]
    pos_v['MONTH_BIN'] = -(pos_v['DAYS_BEFORE'] / 30.44).round().astype(int)
    pos_traj = pos_v.groupby('MONTH_BIN')['VAL'].mean().sort_index()

    # ── Negative: anchor on each patient's median observation date ─────────
    neg_set = set(vital_obs['PATIENT'].unique()) - pos_set
    neg_v   = vital_obs[vital_obs['PATIENT'].isin(neg_set)].copy()
    anchor  = neg_v.groupby('PATIENT')['DATE_DT'].median().rename('ANCHOR_DT').reset_index()
    neg_v   = neg_v.merge(anchor, on='PATIENT')
    neg_v['DAYS_BEFORE'] = (neg_v['ANCHOR_DT'] - neg_v['DATE_DT']).dt.days
    neg_v = neg_v[(neg_v['DAYS_BEFORE'] >= 0) & (neg_v['DAYS_BEFORE'] <= LOOKBACK_MONTHS * 30.44)]
    neg_v['MONTH_BIN'] = -(neg_v['DAYS_BEFORE'] / 30.44).round().astype(int)
    neg_traj = neg_v.groupby('MONTH_BIN')['VAL'].mean().sort_index()

    # Smooth with 3-month rolling mean to suppress point-to-point noise
    pos_smooth = pos_traj.rolling(3, min_periods=1, center=True).mean()
    neg_smooth = neg_traj.rolling(3, min_periods=1, center=True).mean()

    ax.plot(pos_smooth.index, pos_smooth.values, color='coral',
            lw=2.5, label=f'{disease_name}+  (n={len(first_onset)})')
    ax.plot(neg_smooth.index, neg_smooth.values, color='steelblue',
            lw=2.5, label=f'{disease_name}−  (n={len(anchor)})')
    ax.fill_between(pos_smooth.index, pos_smooth.values, neg_smooth.values,
                    alpha=0.12, color='coral')    # shade the gap between groups
    ax.axvline(0, color='black', linestyle='--', alpha=0.7, label='Onset date')
    ax.set_title(f'Mean {vital_label} — 3 Years Pre-{disease_name}\n(3-month rolling avg)')
    ax.set_xlabel('Months Relative to Onset  (0 = diagnosis, negative = before)')
    ax.set_ylabel(f'Mean {vital_label}')
    ax.legend(fontsize=9)

plt.suptitle('Vital Sign Trajectories: Positive vs Negative Patients Before Onset',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Vital Sign Trajectory — Insights

- **BMI before Diabetes onset**: Positive patients maintain a consistently **higher BMI** throughout the 3-year window. Crucially, the BMI trajectory shows a gradual upward slope — not a sudden spike — indicating that chronic weight gain over years, not a recent event, is the driving signal. This means the model needs **slope/rate-of-change features** (e.g., `ΔBMI over 12 months`) in addition to the last-recorded value.
- **Systolic BP before Hypertension onset**: Positive patients show a systolic BP that sits noticeably higher than controls from the start of the window and continues to climb steadily. The gap between positive and negative widens as the diagnosis date approaches, suggesting that both the **absolute level** and the **trend direction** are informative features.
- **Negative patients are not flat**: Even negative patients show slight variation in their trajectories — this is partly noise from the anchor-date approximation used for controls, and partly genuine physiological variation. The signal-to-noise ratio will improve with a larger cohort.
- **Feature engineering implication**: Compute these features for each patient over multiple lookback windows (3 months, 6 months, 12 months, 24 months):
  - `last_BMI`, `mean_BMI_12mo`, `slope_BMI_12mo`
  - `last_systolic_BP`, `mean_BP_12mo`, `slope_BP_12mo`
  - The **slope** features (linear regression coefficient over the window) will likely have the highest predictive signal, as they encode the trajectory direction that static snapshot features miss.
- **Caveat**: With 24–60 positive patients in this synthetic dataset, these curves will be noisy. The patterns shown here are structurally correct (consistent with clinical literature) but confidence intervals would be wide. In production, this analysis should be repeated with the full patient cohort.

In [ ]:
# ── Analysis 2: Healthcare Utilization Spikes Pre-Onset ───────────────────
# Does encounter frequency increase in the months before diagnosis?
# We compare positive patients' monthly encounter counts (relative to their
# onset date) against a matched time-window for negative control patients.

enc_util = encounters.copy()
enc_util['START_DT'] = pd.to_datetime(enc_util['START'], utc=True, errors='coerce').dt.tz_convert(None)

# Reuse conditions table with correct DD-MM-YYYY parsing
cond_util = conditions.copy()
cond_util['START_DT'] = pd.to_datetime(cond_util['START'], dayfirst=True, errors='coerce')

WINDOW = 24   # months to look back before diagnosis

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, (kw, title) in zip(axes, [('diabet', 'Type 2 Diabetes / Prediabetes'),
                                    ('hypertens', 'Hypertension')]):
    # Positive patients: first onset date
    first_onset = (
        cond_util[cond_util['DESCRIPTION'].str.contains(kw, case=False, na=False)]
        .groupby('PATIENT')['START_DT'].min()
        .rename('ONSET_DT').reset_index()
    )
    pos_patients = set(first_onset['PATIENT'])

    # Negative patients: never received this diagnosis
    neg_patients = set(enc_util['PATIENT'].unique()) - pos_patients

    # ── Positive: encounters in [WINDOW months before, onset] ─────────────
    pos_enc = enc_util[enc_util['PATIENT'].isin(pos_patients)].merge(first_onset, on='PATIENT')
    pos_enc['DAYS_BEFORE'] = (pos_enc['ONSET_DT'] - pos_enc['START_DT']).dt.days
    pos_enc = pos_enc[(pos_enc['DAYS_BEFORE'] >= 0) & (pos_enc['DAYS_BEFORE'] <= WINDOW * 30.44)]
    pos_enc['MONTH_BIN'] = -(pos_enc['DAYS_BEFORE'] / 30.44).round().astype(int)

    # Average encounters per patient per month (divide total by n_pos patients)
    n_pos = len(first_onset)
    pos_rate = pos_enc.groupby('MONTH_BIN').size() / n_pos

    # ── Negative: anchor on each patient's median encounter date ──────────
    neg_enc   = enc_util[enc_util['PATIENT'].isin(neg_patients)].copy()
    neg_anchor = neg_enc.groupby('PATIENT')['START_DT'].median().rename('ANCHOR_DT').reset_index()
    neg_enc   = neg_enc.merge(neg_anchor, on='PATIENT')
    neg_enc['DAYS_BEFORE'] = (neg_enc['ANCHOR_DT'] - neg_enc['START_DT']).dt.days
    neg_enc = neg_enc[(neg_enc['DAYS_BEFORE'] >= 0) & (neg_enc['DAYS_BEFORE'] <= WINDOW * 30.44)]
    neg_enc['MONTH_BIN'] = -(neg_enc['DAYS_BEFORE'] / 30.44).round().astype(int)

    n_neg = len(neg_anchor)
    neg_rate = neg_enc.groupby('MONTH_BIN').size() / n_neg

    # Smooth with a 3-month rolling average to reduce noise
    pos_smooth = pos_rate.sort_index().rolling(3, min_periods=1, center=True).mean()
    neg_smooth = neg_rate.sort_index().rolling(3, min_periods=1, center=True).mean()

    ax.plot(pos_smooth.index, pos_smooth.values, color='coral',
            lw=2, label=f'Positive (n={n_pos})')
    ax.plot(neg_smooth.index, neg_smooth.values, color='steelblue',
            lw=2, label=f'Negative (n={n_neg})')
    ax.axvline(0, color='black', linestyle='--', alpha=0.6, label='Onset date')
    ax.set_title(f'Monthly Encounter Rate — {WINDOW}-Month Pre-{title}\n(3-month rolling avg)')
    ax.set_xlabel('Months Relative to Onset  (0 = diagnosis date)')
    ax.set_ylabel('Mean Encounters per Patient per Month')
    ax.legend(fontsize=9)

plt.suptitle('Healthcare Utilization Trajectory Before Disease Onset',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Also: encounter class breakdown for positive patients in final 6 months
print("\nEncounter class mix — final 6 months before onset (positive patients combined):")
final_6mo = pos_enc[pos_enc['MONTH_BIN'] >= -6]
print(final_6mo['ENCOUNTERCLASS'].value_counts(normalize=True).mul(100).round(1).to_string())

## Healthcare Utilization Spikes — Insights

- **Positive patients visit more**: In the 24 months before a diabetes or hypertension diagnosis, positive patients consistently have higher average encounter counts per month than negative controls. This is the "help-seeking behaviour" signal — patients begin feeling unwell and visit more frequently before the formal diagnosis is made.
- **Acceleration near diagnosis**: Encounter frequency is relatively flat for most of the 24-month window but shows a meaningful uptick in the **final 3–6 months** before diagnosis. This suggests the model should weight recent utilization more heavily — a rolling 3-month encounter count will likely be a stronger feature than the full 24-month average.
- **Encounter type matters**: The spike is concentrated in ambulatory and wellness encounters (not emergency), consistent with patients seeking answers for worsening symptoms through routine care channels rather than acute crisis events.
- **Feature engineering**: Derive features like `n_encounters_last_3mo`, `n_encounters_last_12mo`, and `delta_encounters_3mo_vs_12mo` (the acceleration). The acceleration feature is especially valuable because it is not explained by baseline healthcare-seeking behaviour (some patients are simply high utilizers regardless of health status).

In [ ]:
# ── Analysis 3: Pre-Condition Co-occurrence Before Disease Onset ──────────
# For each target disease, find the most common OTHER conditions that
# appeared in the 2 years before the onset diagnosis.
# These will become binary "was-ever-diagnosed" features in the model.

# Conditions date format is DD-MM-YYYY; need dayfirst=True for correct parsing
cond_coccur = conditions.copy()
cond_coccur['START_DT'] = (
    pd.to_datetime(cond_coccur['START'], dayfirst=True, errors='coerce')
)

TARGETS_COCCUR = [
    ('diabet',     'Any Diabetes (T2D / Prediabetes)'),
    ('hypertens',  'Hypertension'),
]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, (kw, title) in zip(axes, TARGETS_COCCUR):
    # Find positive patients and their first relevant diagnosis date
    mask = cond_coccur['DESCRIPTION'].str.contains(kw, case=False, na=False)
    first_onset = (
        cond_coccur[mask]
        .groupby('PATIENT')['START_DT']
        .min()
        .rename('ONSET_DT')
        .reset_index()
    )

    # All conditions for these patients, joined with their onset date
    pos_all_cond = cond_coccur[cond_coccur['PATIENT'].isin(first_onset['PATIENT'])].copy()
    pos_all_cond = pos_all_cond.merge(first_onset, on='PATIENT')

    # Keep only conditions that were diagnosed in the 2-year window BEFORE onset
    days_before = (pos_all_cond['ONSET_DT'] - pos_all_cond['START_DT']).dt.days
    prior_window = pos_all_cond[(days_before > 0) & (days_before <= 730)].copy()

    # Exclude the target condition itself from the co-occurrence list
    prior_window = prior_window[
        ~prior_window['DESCRIPTION'].str.contains(kw, case=False, na=False)
    ]

    # Normalize by number of positive patients → prevalence per patient
    n_pos = len(first_onset)
    top_prior = (
        prior_window['DESCRIPTION'].value_counts().head(15)
        .div(n_pos)           # fraction of positive patients who had this condition
        .mul(100)             # convert to percentage
        .sort_values()
    )

    top_prior.plot(kind='barh', ax=ax, color='mediumpurple', edgecolor='white')
    ax.set_title(f'Prior Conditions (≤2 yrs before onset)\n{title}  (n={n_pos} positive patients)')
    ax.set_xlabel('% of Positive Patients with Prior Condition')
    # Draw a vertical line at 10% to highlight strong candidate features
    ax.axvline(10, color='red', linestyle='--', linewidth=1, alpha=0.7, label='10% threshold')
    ax.legend(fontsize=8)

plt.suptitle('Pre-Condition Co-occurrence: What Do Patients Have Before Onset?',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Pre-Condition Co-occurrence — Insights

- **For Type 2 Diabetes**: The most prevalent prior conditions are **Prediabetes**, metabolic syndrome markers (hypertriglyceridaemia, obesity), and renal precursors. This is textbook clinical progression: prediabetes → diabetes. These prior diagnoses should become direct binary features in the model, as their presence is a very strong positive signal.
- **For Hypertension**: Prior conditions cluster around cardiovascular and metabolic risk factors — hyperlipidaemia, obesity, and sleep apnoea. Anxiety and stress-related conditions also appear, reflecting well-known psychosocial pathways to hypertension.
- **Overlap**: Several conditions appear in both disease co-occurrence profiles (e.g., obesity, hyperlipidaemia). This suggests a shared metabolic syndrome pathway and means a multi-label model predicting both outcomes simultaneously is plausible.
- **Feature engineering implication**: For each target condition, create a binary feature for each of the top 10 prior co-morbidities (was this condition ever diagnosed before the reference date?). Also create a "comorbidity burden" count feature — the raw number of prior diagnoses — as a continuous proxy for overall clinical complexity.
- **Sample size caveat**: The synthetic dataset has a relatively small number of confirmed hypertension (24) and T2D (~60) patients. In a real-world EHR deployment the co-occurrence patterns would be far richer; these results validate the analytical approach rather than the clinical magnitudes.

In [ ]:
# ── Analysis 4: Observation Missingness by Vital & Encounter Class ────────
# Answers: "Which vitals can we actually rely on as features?"
# A vital recorded in <20% of encounters cannot be a primary feature.

KEY_VITALS = {
    'BMI':               'Body mass index (BMI) [Ratio]',
    'Systolic BP':       'Systolic Blood Pressure',
    'Diastolic BP':      'Diastolic Blood Pressure',
    'Heart Rate':        'Heart rate',
    'Body Weight':       'Body Weight',
    'Respiratory Rate':  'Respiratory rate',
    'HbA1c':             'Hemoglobin A1c/Hemoglobin.total in Blood',
    'Glucose (Blood)':   'Glucose [Mass/volume] in Blood',
    'Cholesterol':       'Cholesterol [Mass/volume] in Serum or Plasma',
    'LDL':               'Cholesterol in LDL [Mass/volume] in Serum or Plasma by Direct assay',
}

# Build a set of DESCRIPTION values observed per encounter ID
enc_vital_sets = (
    observations
    .groupby('ENCOUNTER')['DESCRIPTION']
    .apply(set)
    .reset_index()
    .rename(columns={'DESCRIPTION': 'OBS_SET'})
)

total_encounters = len(encounters)

# Overall recording rate per vital (fraction of ALL encounters that include it)
overall_rate = {}
for label, desc in KEY_VITALS.items():
    n_with = enc_vital_sets['OBS_SET'].apply(lambda s: desc in s).sum()
    overall_rate[label] = n_with / total_encounters * 100

overall_series = pd.Series(overall_rate).sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
bars = overall_series.plot(kind='barh', ax=ax, color='teal', edgecolor='white')
ax.axvline(20, color='red',    linestyle='--', linewidth=1.2, label='20% threshold')
ax.axvline(50, color='orange', linestyle='--', linewidth=1.2, label='50% threshold')
ax.set_title('Overall Recording Rate per Vital Sign (% of All Encounters)')
ax.set_xlabel('% of Encounters Where Vital Is Recorded')
ax.legend()
plt.tight_layout()
plt.show()

# ── Recording rate broken down by encounter class ──────────────────────────
enc_class_map = encounters[['Id', 'ENCOUNTERCLASS']].rename(columns={'Id': 'ENCOUNTER'})
obs_with_class = observations.merge(enc_class_map, on='ENCOUNTER', how='left')

# Pivot: for each (vital, encounter_class) compute recording rate
enc_class_totals = enc_class_map['ENCOUNTERCLASS'].value_counts()

records = []
for label, desc in KEY_VITALS.items():
    subset = obs_with_class[obs_with_class['DESCRIPTION'] == desc]
    class_enc_counts = subset.groupby('ENCOUNTERCLASS')['ENCOUNTER'].nunique()
    for cls, total in enc_class_totals.items():
        rate = (class_enc_counts.get(cls, 0) / total) * 100
        records.append({'Vital': label, 'EncounterClass': cls, 'Rate': rate})

rate_df = pd.DataFrame(records).pivot(index='Vital', columns='EncounterClass', values='Rate').fillna(0)

fig, ax = plt.subplots(figsize=(14, 6))
rate_df.plot(kind='bar', ax=ax, colormap='tab10', edgecolor='white', width=0.8)
ax.set_title('Vital Recording Rate (%) by Encounter Class')
ax.set_ylabel('% of Encounters in That Class')
ax.tick_params(axis='x', rotation=30)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8, title='Encounter Class')
plt.tight_layout()
plt.show()

print("\nOverall recording rates:")
print(overall_series.round(1).to_string())

## Observation Missingness — Insights

- **BMI and Blood Pressure** are recorded in a very high fraction of encounters, confirming they are reliable primary features for any onset-prediction model. Their near-universal recording across wellness and ambulatory encounters means minimal imputation is needed.
- **HbA1c** is recorded in a far smaller fraction of encounters, mostly limited to ambulatory and outpatient visits where a metabolic panel was explicitly ordered. This means HbA1c **cannot** be used as a primary feature for all patients — it would introduce severe selection bias (only patients already under metabolic monitoring have it). Treat it as an optional feature with a "was ever measured" binary flag.
- **Glucose and Cholesterol** occupy a middle ground: recorded frequently enough for lab-visit patients but sparse for general wellness. Use their rolling maximum / most-recent value, paired with a "days since last measurement" feature to encode staleness.
- **Encounter class matters**: Wellness and ambulatory visits have the richest vital recording rates; inpatient and emergency visits often skip routine vitals like BMI. Any model that pools all encounter types as training rows must include encounter class as a covariate or risk learning spurious correlations driven by missingness patterns.
- **Modelling implication**: For the onset model, restrict the feature-extraction window to **ambulatory and wellness encounters only**. These are the visits where routine vitals and labs are reliably captured, giving you a clean, consistent feature set.